# CWF Assignment Submission — AY2026 Term 2

> **Important:** Replace all `<< … >>` placeholders before submitting.

---

## Student & CWF Details

- **Term:** AY2026 Term 2
- **Module:** C1339C: Programming for Artificial Intelligence
- **Student Name:** `<<YOUR NAME>>`
- **Student ID:** `<<YOUR STUDENT ID>>`
- **File Name:** `CWF_C1339C_<StudentID>_<StudentName>.ipynb`
  > *Example:* `CWF_C1339C_12345678_PeterKenny.ipynb`

---

## Assessment Overview

| Component | Base Marks | Bonus | Max |
|-----------|-----------|-------|-----|
| Q1: Load and prepare HDB data | 20 | — | 20 |
| Q2: Analyse data | 30 | +5 | 35 |
| Q3: Visualise data | 20 | +5 | 25 |
| Q4: Reflective report | 20 | — | 20 |
| **Total** | **90** | **+10** | **100** |

**Pass mark:** 50/100

### General Grading Principles

Your work will be assessed on:
- **Functionality:** Does your code run correctly from top to bottom?
- **Code Quality:** Is your code well-structured, documented, and PEP 8 compliant?
- **Problem Solving:** Did you demonstrate understanding and good design decisions?
- **Analysis & Insight:** Are your findings meaningful and well-explained?
- **Professional Presentation:** Is your work organised and clearly communicated?

---

## Code Style Requirements (PEP 8)

1. **Indentation:** 4 spaces per level (not tabs)
2. **Line length:** ≤ 79 characters for code, ≤ 72 for comments/docstrings
3. **Imports:** One per line, grouped together at the top of your code sections
4. **Whitespace:** No extra spaces inside `()`, `[]`, `{}` or before `,`, `;`, `:`
5. **Naming:** Functions/variables use `snake_case`, constants use `UPPER_CASE`
6. **Docstrings:** Required for every function *you write* (summary, Args, Returns)

---

## Declaration

I declare that:

1. This submission is entirely my own work; I have not copied or plagiarised any part of it.
2. I have retained a copy of this submission for my records.
3. I have properly cited and acknowledged all sources used.
4. I have neither given nor received unauthorised assistance in completing this work.
5. I have adhered to institutional rules on academic integrity.
6. **Generative AI Usage** *(replace the empty box `[ ]` with `[x]` next to the option that applies — pick exactly one)*:
   - [ ] I **have** used generative AI tools (e.g., ChatGPT, GitHub Copilot, Gemini)
   - [ ] I **have not** used generative AI tools

> **⚠️ WARNING:** You **must** check **ONE** box above regarding AI tool usage before submission. Failure to declare AI usage is a violation of academic integrity policy.

I understand that academic dishonesty may result in disciplinary action.

**Date (DD MMM YYYY):** ________________

---

## Environment Setup

Before running this notebook, ensure your Python environment has all
required packages installed. Follow the steps below in **PowerShell**.

---

### Step 1 — Activate the course environment

```powershell
conda activate ai-course
```

---

### Step 2 — Install required packages

Using **conda** (recommended):

```powershell
conda install numpy pandas matplotlib seaborn statsmodels -y
```

Or, if you prefer **pip**:

```powershell
pip install numpy pandas matplotlib seaborn statsmodels
```

> **Note:** If `conda activate` gives an error, run
> `conda init powershell` first, then close and reopen PowerShell.

## Provided: Data Fetch Utility

The cell below is provided for you — **do not modify it**. It defines
a function called `fetch_town()` that downloads HDB resale price records
directly from the Singapore government's open data API.

**Before starting Question 1, run this cell** (click on it and press
**Shift + Enter**). You will see `fetch_town() is ready.` printed when
it has loaded successfully.

### How `fetch_town()` works

- You call it with a town name, e.g. `fetch_town('TAMPINES')`.
- It downloads all resale transactions from **July 2022 to the present**
  for that town from data.gov.sg.
- It returns a **list of dictionaries** — one dictionary per transaction.
- **First call:** downloads from the internet (20–40 seconds). A local
  cache file is saved automatically so that re-running the cell is
  instant.
- **If the server is busy:** the function waits and retries automatically.
  Leave it running — do not interrupt it.
- **Wrong town name:** the function raises a `ValueError` and prints
  a message. Check your spelling and try again.

### Valid town names (must be UPPERCASE and spelled exactly as shown)

> ANG MO KIO, BEDOK, BISHAN, BUKIT BATOK, BUKIT MERAH, BUKIT PANJANG,
> BUKIT TIMAH, CENTRAL AREA, CHOA CHU KANG, CLEMENTI, GEYLANG, HOUGANG,
> JURONG EAST, JURONG WEST, KALLANG/WHAMPOA, MARINE PARADE, PASIR RIS,
> PUNGGOL, QUEENSTOWN, SEMBAWANG, SENGKANG, SERANGOON, TAMPINES,
> TOA PAYOH, WOODLANDS, YISHUN

In [ ]:
# Provided: Data Fetch Utility
# Run this cell once before starting. Do not modify it.

import csv
import io
import json
import time
import requests
from datetime import datetime
from pathlib import Path

_HDB_RESOURCE_ID = 'd_8b84c4ee58e3cfc0ece0d773c8ca6abc'
_BASE = (
    'https://api-open.data.gov.sg/v1/public/api/datasets/'
    + _HDB_RESOURCE_ID
)
_INITIATE_URL = _BASE + '/initiate-download'
_POLL_URL = _BASE + '/poll-download'
_CACHE_DIR = Path('hdb_cache')
_START_DATE = '2022-07'


def fetch_town(town, start_date=_START_DATE):
    '''
    Fetch HDB resale flat price records for one Singapore town.

    Returns all records from start_date to the present, newest first.
    After the first download, results are read from a local cache file
    so subsequent calls return instantly.

    Args:
        town (str): Town name. Case-insensitive. Valid values:
            ANG MO KIO, BEDOK, BISHAN, BUKIT BATOK, BUKIT MERAH,
            BUKIT PANJANG, BUKIT TIMAH, CENTRAL AREA, CHOA CHU KANG,
            CLEMENTI, GEYLANG, HOUGANG, JURONG EAST, JURONG WEST,
            KALLANG/WHAMPOA, MARINE PARADE, PASIR RIS, PUNGGOL,
            QUEENSTOWN, SEMBAWANG, SENGKANG, SERANGOON, TAMPINES,
            TOA PAYOH, WOODLANDS, YISHUN.
        start_date (str): Earliest month to include, as 'YYYY-MM'.
            Default: '2022-07'.

    Returns:
        list[dict]: Resale transaction records, newest first.

    Raises:
        ValueError: If the town name does not match any records.
    '''
    town_upper = town.strip().upper()
    _CACHE_DIR.mkdir(exist_ok=True)
    cache_file = _CACHE_DIR / (town_upper.replace(' ', '_') + '.json')

    if cache_file.exists():
        with open(cache_file, encoding='utf-8') as f:
            stored = json.load(f)
        if stored.get('start_date') != start_date:
            print(f'  Cache outdated — re-downloading ...')
            cache_file.unlink()
        else:
            records = stored['records']
            cached_at = stored['cached_at']
            print(f'Loaded {len(records)} {town_upper} records '
                  f'from cache (saved {cached_at})')
            return records

    print(f'Fetching {town_upper} from data.gov.sg ...')
    payload = {
        'filters': [
            {'columnName': 'town', 'type': 'EQ', 'value': town_upper}
        ]
    }

    def _request(url, json_body=None):
        while True:
            try:
                resp = requests.get(url, json=json_body, timeout=30)
            except requests.exceptions.RequestException as exc:
                print(f'  Network error: {exc} — retrying in 30 s ...')
                time.sleep(30)
                continue
            if resp.status_code == 429:
                print('  Rate limit — waiting 20 s ...')
                time.sleep(20)
                continue
            resp.raise_for_status()
            return resp

    _request(_INITIATE_URL, json_body=payload)

    download_url = None
    while not download_url:
        time.sleep(5)
        data = _request(_POLL_URL, json_body=payload).json().get(
            'data', {}
        )
        download_url = data.get('url')
        if not download_url:
            print('  Preparing download ...')

    file_resp = requests.get(download_url, timeout=30)
    file_resp.raise_for_status()

    ct = file_resp.headers.get('content-type', '').lower()
    if 'json' in ct:
        raw = file_resp.json()
        if isinstance(raw, list):
            rows = raw
        elif isinstance(raw, dict):
            rows = (raw.get('records')
                    or raw.get('result', {}).get('records', [])
                    or [])
        else:
            rows = []
    else:
        rows = list(csv.DictReader(io.StringIO(file_resp.text)))

    if not rows:
        raise ValueError(
            'No records found for town ' + repr(town) + '. '
            'Check spelling — e.g. TAMPINES, QUEENSTOWN.'
        )

    records = [
        r for r in rows if r.get('month', '') >= start_date
    ]
    records.sort(
        key=lambda r: (r.get('month', ''), r.get('_id', '')),
        reverse=True
    )

    with open(cache_file, 'w', encoding='utf-8') as f:
        json.dump(
            {
                'town': town_upper,
                'start_date': start_date,
                'cached_at': datetime.now().strftime('%Y-%m-%d %H:%M'),
                'records': records,
            },
            f,
            indent=2,
        )

    print(f'  Done — {len(records)} records saved to {cache_file}')
    return records


print('fetch_town() is ready.')
print('See the function docstring for the list of valid town names.')

## Your Imports

Add all your import statements in the cell below. Some suggested packages
you will need are already included. Add any others you use.

> **PEP 8 reminder:** one import per line, all imports at the top.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
# Add any additional imports here

---

## Question 1: Load and Prepare HDB Resale Data (20 marks)

Use the provided `fetch_town()` function to download HDB resale price
data for **two Singapore towns of your choice**, then prepare it for
analysis.

### Requirements

- Call `fetch_town()` for two different towns. Read the function's
  docstring to understand what it accepts and what it returns.
- Convert the results into pandas DataFrames. All values returned by
  the function are strings — you must convert the relevant columns to
  appropriate data types before any analysis.
- Display summary statistics for both DataFrames.

### What to submit

- Working code in the cell below
- Visible output showing both DataFrames' summaries
- A brief markdown note (2–3 sentences) naming your chosen towns and
  commenting on anything you observe in the summaries

In [ ]:
# Q1 — Your code here


---

## Question 2: Analyse Data (30 marks + 5 bonus marks)

Using your two DataFrames from Question 1, perform a statistical
analysis of HDB resale prices.

---

### Task 1 — Town price summary

For **each town**, calculate and display the mean, median, and standard
deviation of `resale_price`.

You must use **NumPy** for at least one of these calculations.

Present the results clearly — a printed summary or a small table.

---

### Task 2 — Price by flat type

For **each town**, group the data by `flat_type` and calculate the
mean and median `resale_price` for each flat type. Display the results
as a readable table.

---

### Task 3 — Written comparison

In the markdown placeholder cell below your code, write a comparison
of the two towns. Include at least **three specific observations**
supported by figures from your analysis.

---

**Bonus (+5 marks):** Calculate price per square metre and add it as
a new column to each DataFrame. Then show how it varies by flat type.

In [ ]:
# Q2 — Your code here


### Written Comparison

*Replace this text with your written comparison of the two towns.
Include at least three specific observations supported by figures
from your analysis above. (Aim for 3–5 sentences.)*

---

## Question 3: Visualise Data (20 marks + 5 bonus marks)

Create a professional chart using `matplotlib` that shows how HDB
resale prices have changed over time.

---

### Your chart must:

- Plot monthly median `resale_price` against `month` for one of your chosen
  towns, showing at least **two flat types** as separate series
- Each series must show:
  - Scatter dots at the actual monthly median values
  - A smooth LOWESS trend line through the dots
    (`from statsmodels.nonparametric.smoothers_lowess import lowess`)
- Include a descriptive **title**, **axis labels**, and a **legend**
- Format the y-axis as dollar values (e.g. `$400,000`)
- Be clean, readable, and professionally styled

---

**Bonus (+5 marks):** Add a second chart comparing **both towns** for
a single flat type, using `seaborn.lineplot` with a **95% confidence
interval** (`estimator='median'`, `errorbar=('ci', 95)`).

In [ ]:
# Q3 — Your code here


### Interpretation

*Replace this text with your interpretation of the chart(s). What
patterns or trends do you see? Are any results surprising? Which flat
types have changed the most over time? (2–4 sentences.)*

---

## Question 4: Reflective Report (20 marks)

Write a reflective report of **250–500 words** addressing all four
points below. Use markdown formatting to structure your report
(headings, bullet points, bold text).

1. **Problem-solving approach** — How did you break down and tackle
   this assignment? What was your overall strategy?

2. **Use of Python packages** — Which packages did you use and why?
   How did each one help you solve the problem?

3. **Code quality and structure** — How did you ensure your code was
   well-organised and readable? What decisions did you make about
   naming, structure, or comments?

4. **Data insights and learning** — What did the data reveal about
   Singapore housing prices? Were any findings surprising? What
   challenges did you face and how did you overcome them?

---

## My Reflective Report

*Write your 250–500 word report here. Use headings, bullet points,
and other markdown formatting as appropriate. Address all four points
listed in the question.*

---

## Pre-Submission Checklist

Before submitting, confirm each item below:

- [ ] Student name and ID filled in at the top
- [ ] AI usage declaration checked (exactly one box marked)
- [ ] Date added
- [ ] File renamed to: `CWF_C1339C_<StudentID>_<StudentName>.ipynb`
- [ ] All code cells execute without errors, top to bottom
  (use **Kernel → Restart & Run All** to verify)
- [ ] All functions *you wrote* have docstrings
- [ ] Code follows PEP 8 style guidelines
- [ ] Reflective report is 250–500 words
- [ ] All placeholder text removed
- [ ] Written comparison (Q2) and interpretation (Q3) are completed

---

## Resources

- [HDB Resale Flat Prices dataset](https://data.gov.sg/datasets/d_8b84c4ee58e3cfc0ece0d773c8ca6abc/view)
- [data.gov.sg API Guide](https://guide.data.gov.sg/developers/api-v2)
- [PEP 8 Style Guide](https://pep8.org/)
- [pandas Documentation](https://pandas.pydata.org/docs/)
- [NumPy Documentation](https://numpy.org/doc/)
- [matplotlib Documentation](https://matplotlib.org/)

---

*Good luck!*